# Zero-shot evaluation on BSP and BSP-augmented: side by side

All six models were trained **only on the crowdsourced Google Earth dataset** (8868 / 999 / 999) with identical settings. No Bengaluru image is used for training anywhere in this notebook; it only runs inference.

| Model | Checkpoint | Google test IoU (in-domain) |
|---|---|---:|
| ResNet-UNet (baseline) | `CD_ResnetUnet.pth` | 0.8518 |
| Axial | `CD_ResnetUnet_Axial.pth` | 0.8526 |
| Axial + Transformer | `CD_ResnetUnet_AxialTransformer.pth` | 0.8472 |
| Boundary Head | `CD_ResnetUnet_BoundaryHead.pth` | 0.8530 |
| Boundary-Guided Attention | `CD_ResnetUnet_BoundaryGuidedAttention.pth` | 0.8498 |
| StyleGate | `CD_ResnetUnet_StyleGate.pth` | 0.8466 |

**The two evaluation sets**

| Set | Images | What it contains |
|---|---:|---|
| **BSP** | 43 | every annotated Bengaluru image (Haralur 16, HSR 13, Krystal 14) |
| **BSP augmented** | 183 | the same 43 originals **plus 140 transformed copies of the 20 training originals**: h-flip, v-flip, three small rotations (4-12 deg, black corners), Gaussian blur, and an exact duplicate (`_orig`) |

BSP augmented therefore contains **no new scenes**. Its score is weighted towards the 20 training originals (each counted 8 times), and it measures robustness to flips, rotations and blur. Section 6 separates the two effects: it compares each augmentation type against **the same 20 source images**.

**Sections**
1-4. setup, loading, helpers
5. zero-shot, side by side (main result)
6. per-augmentation breakdown
7. label-free test-time methods (TTA, AdaBN), side by side
8. input-scale sweep, side by side (diagnoses a panel-size gap)
9. diagnostics: panel-level recall by size, false-positive blobs, Google vs Bengaluru statistics, per-image IoU
10. overlays

**Inputs to attach on Kaggle (Add Input):**
- the `.pth` checkpoints: the five from `Final/Results`, **plus `CD_ResnetUnet.pth` (the baseline) if you want the baseline row**. Missing checkpoints are skipped;
- the BSP split: `train/ val/ test/`, each with `Img/` and `Masks/` (20 / 10 / 13);
- BSP augmented: same layout (160 / 10 / 13);
- optional: the crowdsourced Google dataset (only for the Google vs Bengaluru statistics in 9b).

Paths are detected automatically; set them in the configuration cell if detection fails. GPU recommended; total runtime is a few minutes.

In [ ]:
import os, re, copy, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
from scipy import ndimage
from IPython.display import display

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print(f'Using device: {device}')

## 1. Model definitions (identical to the training notebooks)

In [ ]:
# Model definitions copied verbatim from the training notebooks:
#   ResNetUNet                -> CD_UnetResnet.ipynb
#   AxialAttention2d, AxialResNetUNet, BottleneckTransformer, AxialTransResNetUNet -> CD_ResnetUnet_AxialTransformer.ipynb
#   BoundaryHeadResNetUNet    -> CD_ResnetUnet_BoundaryHead.ipynb
#   BoundaryGuidedResNetUNet  -> CD_ResnetUnet_BoundaryGuidedAttention.ipynb
#   StyleGate, StyleGateResNetUNet -> CD_ResnetUnet_StyleGate.ipynb

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_rate=0.15, use_dropout=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))

        layers.extend([
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ])
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))

        self.double_conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.double_conv(x)

# (constructor, ImageNet weights enum name, channels of [stem, layer1, layer2, layer3, layer4])
RESNET_CFG = {
    'resnet18':  (models.resnet18,  'ResNet18_Weights',  [64, 64, 128, 256, 512]),
    'resnet34':  (models.resnet34,  'ResNet34_Weights',  [64, 64, 128, 256, 512]),
    'resnet50':  (models.resnet50,  'ResNet50_Weights',  [64, 256, 512, 1024, 2048]),
    'resnet101': (models.resnet101, 'ResNet101_Weights', [64, 256, 512, 1024, 2048]),
}

class ResNetUNet(nn.Module):
    """U-Net with a ResNet encoder.

    Encoder (strides relative to the input):
        stem   (conv7x7 + BN + ReLU)  /2
        layer1 (after maxpool)        /4
        layer2                        /8
        layer3                        /16
        layer4 (bottleneck)           /32
    Decoder: ConvTranspose2d x2 + skip concat + DoubleConv, five times, back to full resolution.
    Input height/width must be divisible by 32 (256x256 is fine).
    """
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1,
                 dropout_rates=None):
        super().__init__()
        if backbone not in RESNET_CFG:
            raise ValueError(f"backbone must be one of {list(RESNET_CFG)}")

        if dropout_rates is None:
            dropout_rates = {
                'encoder': {'layer3': 0.15, 'layer4': 0.2},   # same idea as U-Net: enc4 = 0.15, bottleneck = 0.2
                'decoder': 0.15,
            }
        self.dropout_rates = dropout_rates

        ctor, weights_name, ch = RESNET_CFG[backbone]
        weights = getattr(models, weights_name).IMAGENET1K_V1 if pretrained else None
        backbone_net = ctor(weights=weights)

        # Optional: support in_ch != 3 by rebuilding the first conv
        if in_ch != 3:
            backbone_net.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Encoder
        self.stem = nn.Sequential(backbone_net.conv1, backbone_net.bn1, backbone_net.relu)  # /2
        self.pool = backbone_net.maxpool                                                    # /4
        self.layer1 = backbone_net.layer1
        self.layer2 = backbone_net.layer2
        self.layer3 = backbone_net.layer3
        self.layer4 = backbone_net.layer4

        self.drop3 = nn.Dropout2d(dropout_rates['encoder']['layer3'])
        self.drop4 = nn.Dropout2d(dropout_rates['encoder']['layer4'])

        c0, c1, c2, c3, c4 = ch
        d = dropout_rates['decoder']

        # Decoder
        self.up4 = nn.ConvTranspose2d(c4, c3, kernel_size=2, stride=2)       # /32 -> /16
        self.dec4 = DoubleConv(c3 * 2, c3, d, use_dropout=True)
        self.up3 = nn.ConvTranspose2d(c3, c2, kernel_size=2, stride=2)       # /16 -> /8
        self.dec3 = DoubleConv(c2 * 2, c2, d, use_dropout=True)
        self.up2 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)       # /8  -> /4
        self.dec2 = DoubleConv(c1 * 2, c1, d, use_dropout=True)
        self.up1 = nn.ConvTranspose2d(c1, c0, kernel_size=2, stride=2)       # /4  -> /2
        self.dec1 = DoubleConv(c0 * 2, c0, d, use_dropout=True)
        self.up0 = nn.ConvTranspose2d(c0, 32, kernel_size=2, stride=2)       # /2  -> full res
        self.dec0 = DoubleConv(32, 32, d, use_dropout=True)

        self.final_conv = nn.Conv2d(32, out_ch, kernel_size=1)

    def forward(self, x):
        # Encoder
        e0 = self.stem(x)                       # /2
        e1 = self.layer1(self.pool(e0))         # /4
        e2 = self.layer2(e1)                    # /8
        e3 = self.drop3(self.layer3(e2))        # /16
        b = self.drop4(self.layer4(e3))         # /32 (bottleneck)

        # Decoder
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))

        return self.final_conv(d0)

# =====================================================================
# Axial attention (same module as BSP_augmented/Final/Resnet_Axial.ipynb)
# =====================================================================
class AxialAttention2d(nn.Module):
    """Axial self-attention: attend along W (rows), then along H (columns).
    Cost O(HW(H+W)) instead of O((HW)^2). Residual with zero-initialised gate, so at
    init the block is identity and the pretrained ResNet-UNet behaviour is preserved."""
    def __init__(self, dim, heads=None, attn_drop=0.0):
        super().__init__()
        self.heads = heads if heads is not None else max(1, dim // 32)
        assert dim % self.heads == 0
        self.pos = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)       # size-agnostic positional encoding
        self.norm = nn.GroupNorm(1, dim)
        self.qkv_w = nn.Conv2d(dim, dim * 3, 1, bias=False)
        self.qkv_h = nn.Conv2d(dim, dim * 3, 1, bias=False)
        self.proj_w = nn.Conv2d(dim, dim, 1)
        self.proj_h = nn.Conv2d(dim, dim, 1)
        self.attn_drop = attn_drop
        self.gamma = nn.Parameter(torch.zeros(1))

    def _attend(self, x, qkv_layer, proj_layer, axis):
        B, C, H, W = x.shape
        q, k, v = qkv_layer(x).chunk(3, dim=1)
        h = self.heads
        def to_seq(t):                                                  # -> (B*other, heads, L, C/heads)
            if axis == 'w':                                             # sequences run along W
                t = t.permute(0, 2, 3, 1).reshape(B * H, W, h, C // h)
            else:                                                       # sequences run along H
                t = t.permute(0, 3, 2, 1).reshape(B * W, H, h, C // h)
            return t.transpose(1, 2)
        out = F.scaled_dot_product_attention(
            to_seq(q), to_seq(k), to_seq(v),
            dropout_p=self.attn_drop if self.training else 0.0)
        out = out.transpose(1, 2)
        if axis == 'w':
            out = out.reshape(B, H, W, C).permute(0, 3, 1, 2)
        else:
            out = out.reshape(B, W, H, C).permute(0, 3, 2, 1)
        return proj_layer(out.contiguous())

    def forward(self, x):
        y = self.norm(x + self.pos(x))
        y = y + self._attend(y, self.qkv_w, self.proj_w, 'w')          # row attention
        y = y + self._attend(self.norm(y), self.qkv_h, self.proj_h, 'h')  # column attention
        return x + self.gamma * y


class AxialResNetUNet(ResNetUNet):
    """ResNet-UNet with axial attention on the /4, /8 and /16 skip connections."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        _, c1, c2, c3, _ = RESNET_CFG[backbone][2]
        self.ax1, self.ax2, self.ax3 = AxialAttention2d(c1), AxialAttention2d(c2), AxialAttention2d(c3)

    def _enc(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        return e0, e1, e2, e3, b

    def forward(self, x):
        e0, e1, e2, e3, b = self._enc(x)
        e1, e2, e3 = self.ax1(e1), self.ax2(e2), self.ax3(e3)
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))


# =====================================================================
# Transformer on the bottleneck (same module as BSP_augmented/Final/Resnet_AxialTransformer.ipynb)
# =====================================================================
class BottleneckTransformer(nn.Module):
    """Transformer encoder on the /32 bottleneck tokens (8x8 = 64 tokens at 256x256).
    Projects to a small width first so it stays cheap for any ResNet backbone."""
    def __init__(self, in_ch, dim=256, depth=2, heads=8, dropout=0.1):
        super().__init__()
        self.proj_in = nn.Conv2d(in_ch, dim, 1)
        self.pos = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)       # conditional positional encoding
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth, enable_nested_tensor=False)
        self.proj_out = nn.Conv2d(dim, in_ch, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, _, H, W = x.shape
        t = self.proj_in(x)
        t = t + self.pos(t)
        t = self.encoder(t.flatten(2).transpose(1, 2))                  # (B, HW, dim)
        t = t.transpose(1, 2).reshape(B, -1, H, W)
        return x + self.gamma * self.proj_out(t)


class AxialTransResNetUNet(AxialResNetUNet):
    """Axial attention on the /4, /8, /16 skips + 2-layer Transformer on the /32 bottleneck."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1,
                 dropout_rates=None, tr_dim=256, tr_depth=2, tr_heads=8):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        c4 = RESNET_CFG[backbone][2][4]
        self.bottleneck_tr = BottleneckTransformer(c4, tr_dim, tr_depth, tr_heads)

    def forward(self, x):
        e0, e1, e2, e3, b = self._enc(x)
        b = self.bottleneck_tr(b)
        e1, e2, e3 = self.ax1(e1), self.ax2(e2), self.ax3(e3)
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))

# =====================================================================
# Auxiliary boundary head (same head as Final Model/ResNet34_BoundaryHead.ipynb)
# =====================================================================
class BoundaryHeadResNetUNet(ResNetUNet):
    """ResNet-UNet + a small boundary head on the full-resolution decoder features.
    The segmentation path is unchanged; the head only adds a training signal.
    forward() returns (segmentation_logits, boundary_logits)."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        self.boundary_head = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, out_ch, kernel_size=1),
        )

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))
        return self.final_conv(d0), self.boundary_head(d0)

# =====================================================================
# Boundary-guided attention (same module as Final Model/resnet34_boundaryguidedattention.ipynb)
# =====================================================================
class BoundaryGuidedResNetUNet(ResNetUNet):
    """ResNet-UNet + boundary head whose prediction reweights the full-resolution decoder
    features before the segmentation head:  d0 + scale * d0 * sigmoid(boundary).
    scale starts at 0, so training starts from the plain ResNet-UNet.
    forward() returns (segmentation_logits, boundary_logits)."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        self.boundary_feature = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.boundary_head = nn.Conv2d(16, out_ch, kernel_size=1)
        self.boundary_attention_scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))

        boundary_logits = self.boundary_head(self.boundary_feature(d0))
        boundary_attention = torch.sigmoid(boundary_logits)
        refined = d0 + self.boundary_attention_scale * (d0 * boundary_attention)
        return self.final_conv(refined), boundary_logits

# =====================================================================
# StyleGate (same module as Final Model/UsingAttentions/Attentions.ipynb)
# =====================================================================
class StyleGate(nn.Module):
    '''Style-invariant channel gate:
        out = x + gamma * g * (IN(x) - x),   g = sigmoid(MLP([channel mean, channel std]))  (per channel)
    IN(x) is the instance-normalised feature: the per-image colour / brightness / contrast statistics
    are removed, which is mostly what changes between regions (Google Earth vs Bengaluru).
    g lets every channel decide how much of the style-free version it wants.
    gamma starts at 0 -> exact identity at the start of training; the network learns whether to use it.'''
    def __init__(self, ch, reduction=4):
        super().__init__()
        self.inorm = nn.InstanceNorm2d(ch, affine=False)
        hid = max(ch // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(2 * ch, hid), nn.ReLU(inplace=True), nn.Linear(hid, ch))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mu = x.mean(dim=(2, 3))
        sd = x.var(dim=(2, 3), unbiased=False).add(1e-5).sqrt()
        g = torch.sigmoid(self.mlp(torch.cat([mu, sd], dim=1)))[:, :, None, None]
        return x + self.gamma * g * (self.inorm(x) - x)


class StyleGateResNetUNet(ResNetUNet):
    '''ResNet-UNet with a StyleGate after the stem (/2) and after layer1 (/4),
    i.e. where low-level appearance (colour, texture, contrast) is encoded.'''
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        c0, c1, _, _, _ = RESNET_CFG[backbone][2]
        self.style0 = StyleGate(c0)
        self.style1 = StyleGate(c1)

    def forward(self, x):
        e0 = self.style0(self.stem(x))                  # /2
        e1 = self.style1(self.layer1(self.pool(e0)))    # /4
        e2 = self.layer2(e1)                            # /8
        e3 = self.drop3(self.layer3(e2))                # /16
        b = self.drop4(self.layer4(e3))                 # /32
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))

## 2. Configuration and automatic path detection
Leave the paths as `None` to search `/kaggle/input` automatically, or paste the exact folders.

In [ ]:
INPUT_ROOT = '/kaggle/input'
OUT_DIR = '/kaggle/working/bsp_eval'
os.makedirs(OUT_DIR, exist_ok=True)

BSP_ROOT = None      # folder with train/ val/ test/ -> Img/ Masks/   (20 / 10 / 13)
AUG_ROOT = None      # folder with train/ val/ test/ -> Img/ Masks/   (160 / 10 / 13)
CKPT_DIR = None      # folder with the .pth files (None = search all of /kaggle/input)
GOOGLE_ROOT = None   # optional: crowdsourced 'Final' folder (train/ val/ test/ -> img/ mask/), section 9b only

IMG_SIZE = (256, 256)   # same as training
THRESH = 0.5            # same as training
BATCH = 8

# (display name, class, checkpoint file, Google in-domain test IoU from its training notebook)
MODEL_SPECS = [
    ('ResNet-UNet (baseline)',    ResNetUNet,               'CD_ResnetUnet.pth',                         0.8518),
    ('Axial',                     AxialResNetUNet,          'CD_ResnetUnet_Axial.pth',                   0.8526),
    ('Axial + Transformer',       AxialTransResNetUNet,     'CD_ResnetUnet_AxialTransformer.pth',        0.8472),
    ('Boundary Head',             BoundaryHeadResNetUNet,   'CD_ResnetUnet_BoundaryHead.pth',            0.8530),
    ('Boundary-Guided Attention', BoundaryGuidedResNetUNet, 'CD_ResnetUnet_BoundaryGuidedAttention.pth', 0.8498),
    ('StyleGate',                 StyleGateResNetUNet,      'CD_ResnetUnet_StyleGate.pth',               0.8466),
]
GOOGLE_IOU = {name: g for name, _, _, g in MODEL_SPECS}
BASELINE = MODEL_SPECS[0][0]

# same dropout config as training (inactive in eval mode, but the module layout must match the checkpoint)
DROPOUT_CONFIG = {'encoder': {'layer3': 0.15, 'layer4': 0.2}, 'decoder': 0.15}
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')


def ci_sub(parent, name):
    # case-insensitive sub-folder lookup
    if parent is None or not os.path.isdir(parent):
        return None
    for d in os.listdir(parent):
        if d.lower() == name.lower() and os.path.isdir(os.path.join(parent, d)):
            return os.path.join(parent, d)
    return None


def split_dirs(root, img_name, mask_name):
    # {split: (img_dir, mask_dir)} if root has train/ val/ test/ with the given sub-folders, else None
    out = {}
    for s in ('train', 'val', 'test'):
        sd = ci_sub(root, s)
        im, mk = ci_sub(sd, img_name), ci_sub(sd, mask_name)
        if im is None or mk is None:
            return None
        out[s] = (im, mk)
    return out


def n_files(d):
    return sum(f.lower().endswith(IMG_EXTS) for f in os.listdir(d))


all_dirs, pth_files = [], {}
for dp, dns, fns in os.walk(INPUT_ROOT, followlinks=True):
    all_dirs.append(dp)
    for f in fns:
        if f.endswith('.pth'):
            pth_files.setdefault(f, []).append(os.path.join(dp, f))

# ---- checkpoints ----
ckpt_paths = {}
for name, _, fname, _ in MODEL_SPECS:
    if CKPT_DIR is not None and os.path.isfile(os.path.join(CKPT_DIR, fname)):
        ckpt_paths[name] = os.path.join(CKPT_DIR, fname)
    elif fname in pth_files:
        ckpt_paths[name] = sorted(pth_files[fname])[0]
for name, p in ckpt_paths.items():
    print(f'  {name:27s} -> {p}')
missing = [fname for name, _, fname, _ in MODEL_SPECS if name not in ckpt_paths]
if missing:
    print('\nMISSING checkpoints (skipped):', missing)
    print('.pth files visible under /kaggle/input:', sorted(pth_files))
assert ckpt_paths, 'No checkpoints found: attach the dataset with the .pth files or set CKPT_DIR.'

# ---- BSP and BSP augmented (both have Img/ + Masks/; told apart by the number of training images) ----
cands = [d for d in all_dirs if split_dirs(d, 'img', 'masks')]
n_train = {d: n_files(split_dirs(d, 'img', 'masks')['train'][0]) for d in cands}
print('\nsplit folders found (train image count):', {d: n_train[d] for d in cands})
if BSP_ROOT is None:
    c = [d for d in cands if n_train[d] < 60 and 'augment' not in d.lower()] or [d for d in cands if n_train[d] < 60]
    assert c, 'BSP (20/10/13) not found: attach it or set BSP_ROOT.'
    BSP_ROOT = min(c, key=lambda d: abs(n_train[d] - 20))
if AUG_ROOT is None:
    c = [d for d in cands if n_train[d] >= 60]
    assert c, 'BSP augmented (160/10/13) not found: attach it or set AUG_ROOT.'
    AUG_ROOT = min(c, key=lambda d: abs(n_train[d] - 160))
print('BSP_ROOT =', BSP_ROOT)
print('AUG_ROOT =', AUG_ROOT)

# ---- optional Google dataset (img/ + mask/ with *_gt masks) ----
if GOOGLE_ROOT is None:
    for d in all_dirs:
        sd = split_dirs(d, 'img', 'mask')
        if sd and any('_gt' in f for f in os.listdir(sd['test'][1])[:50]):
            GOOGLE_ROOT = d
            break
print('GOOGLE_ROOT =', GOOGLE_ROOT, '' if GOOGLE_ROOT else '(not attached -> 9b shows BSP statistics only)')

## 3. Load both datasets
Same preprocessing as training: resize to 256x256 (PIL default for images, nearest for masks), ImageNet normalisation, mask = pixel > 0. Every image is tagged with its split, layout, augmentation type and source image.

In [ ]:
NORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
AUG_TAGS = ('orig', 'hflip', 'vflip', 'rotA', 'rotB', 'rotC', 'blur')
AUG_LABEL = {'orig': 'duplicate (_orig)', 'hflip': 'h-flip', 'vflip': 'v-flip', 'rotA': 'rotate -12..-4',
             'rotB': 'rotate +4..+12', 'rotC': 'rotate +-4..12', 'blur': 'Gaussian blur'}


def layout_of(stem):
    s = stem.lower()
    if s.startswith('behind_layout') or s.startswith('haralur'):
        return 'Haralur'
    if s.startswith('hsr'):
        return 'HSR'
    if s.startswith('kristal') or s.startswith('krystal'):
        return 'Krystal'
    return 'Other'


def load_dataset(root):
    recs = []
    for split, (img_dir, mask_dir) in split_dirs(root, 'img', 'masks').items():
        masks = {}
        for f in os.listdir(mask_dir):
            if f.lower().endswith(IMG_EXTS):
                st = os.path.splitext(f)[0]
                masks[st[:-5] if st.lower().endswith('_mask') else st] = f
        for f in sorted(os.listdir(img_dir)):
            st = os.path.splitext(f)[0]
            if not f.lower().endswith(IMG_EXTS) or st not in masks:
                continue
            m = re.search(r'_(' + '|'.join(AUG_TAGS) + r')$', st)
            aug, source = (m.group(1), st[:m.start()]) if m else ('original', st)
            img = Image.open(os.path.join(img_dir, f)).convert('RGB').resize(IMG_SIZE)
            lab = Image.open(os.path.join(mask_dir, masks[st])).convert('L').resize(IMG_SIZE, Image.NEAREST)
            recs.append(dict(name=st, split=split, layout=layout_of(st), aug=aug, source=source,
                             x=NORM(img), y=torch.from_numpy(np.array(lab) > 0), rgb=np.array(img)))
    meta = pd.DataFrame([{k: r[k] for k in ('name', 'split', 'layout', 'aug', 'source')} for r in recs])
    Ys = torch.stack([r['y'] for r in recs])
    meta['panel_frac'] = Ys.float().mean(dim=(1, 2)).numpy()
    return dict(X=torch.stack([r['x'] for r in recs]), Y=Ys, RGB=np.stack([r['rgb'] for r in recs]), meta=meta)


idx_of = lambda mask: torch.tensor(np.where(np.asarray(mask))[0], dtype=torch.long)

DS = {'BSP': load_dataset(BSP_ROOT), 'BSP augmented': load_dataset(AUG_ROOT)}
DNAMES = list(DS)

for dname, d in DS.items():
    m = d['meta']
    d['subsets'] = {'All': idx_of(np.ones(len(m), bool)),
                    'Test 13': idx_of(m.split == 'test'),
                    'Haralur': idx_of(m.layout == 'Haralur'),
                    'HSR': idx_of(m.layout == 'HSR'),
                    'Krystal': idx_of(m.layout == 'Krystal')}
    print(f'\n{dname}: {len(m)} image/mask pairs')
    display(pd.crosstab([m.split], m.aug, margins=True))

ma = DS['BSP augmented']['meta']
DS['BSP augmented']['subsets'].update({
    'Originals (43)': idx_of(ma.aug == 'original'),
    'Augmented copies (140)': idx_of(ma.aug != 'original'),
})
if len(DS['BSP']['meta']) != 43 or len(ma) != 183:
    print('WARNING: expected 43 and 183 pairs - check BSP_ROOT / AUG_ROOT.')

## 4. Helpers: model loading, prediction (plain / TTA / rescaled input), AdaBN, metrics

In [ ]:
def build_model(cls, ckpt):
    m = cls(backbone='resnet34', pretrained=False, in_ch=3, out_ch=1, dropout_rates=DROPOUT_CONFIG)
    sd = torch.load(ckpt, map_location='cpu')
    sd = {(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}
    m.load_state_dict(sd, strict=True)
    return m.to(device).eval()


def seg_logits(model, x):
    out = model(x)
    return out[0] if isinstance(out, (tuple, list)) else out   # boundary models return (seg, boundary)


def _dihedral(k, flip):
    fwd = lambda t: torch.rot90(torch.flip(t, dims=[3]) if flip else t, k, dims=(2, 3))
    inv = lambda t: (lambda u: torch.flip(u, dims=[3]) if flip else u)(torch.rot90(t, -k, dims=(2, 3)))
    return fwd, inv

IDENTITY = [_dihedral(0, False)]
DIHEDRAL = [_dihedral(k, f) for f in (False, True) for k in range(4)]   # 8 flips / rotations


@torch.no_grad()
def predict_probs(model, Xs, tta=False, scale=1.0, bs=BATCH):
    # (N,H,W) float16 probabilities at 256x256
    model.eval()
    outs = []
    for i in range(0, len(Xs), bs):
        xb = Xs[i:i + bs].to(device)
        H, W = xb.shape[-2:]
        if scale != 1.0:
            size = (max(32, int(round(H * scale / 32)) * 32), max(32, int(round(W * scale / 32)) * 32))
            xb = F.interpolate(xb, size=size, mode='bilinear', align_corners=False, antialias=scale < 1.0)
        p = 0
        tfs = DIHEDRAL if tta else IDENTITY
        for fwd, inv in tfs:
            p = p + torch.sigmoid(inv(seg_logits(model, fwd(xb))))
        p = p / len(tfs)
        if scale != 1.0:
            p = F.interpolate(p, size=(H, W), mode='bilinear', align_corners=False)
        outs.append(p[:, 0].half().cpu())
    return torch.cat(outs)


@torch.no_grad()
def adabn(model, X_adapt, bs=BATCH):
    # AdaBN: re-estimate every BatchNorm's running mean/var on unlabelled target images
    # (cumulative average over all 8 dihedral views). Weights unchanged, dropout stays off.
    m = copy.deepcopy(model)
    bns = [mod for mod in m.modules() if isinstance(mod, nn.modules.batchnorm._BatchNorm)]
    for bn in bns:
        bn.reset_running_stats()
        bn.momentum = None
    m.eval()
    for bn in bns:
        bn.train()
    for i in range(0, len(X_adapt), bs):
        xb = X_adapt[i:i + bs].to(device)
        for fwd, _ in DIHEDRAL:
            seg_logits(m, fwd(xb))
    return m.eval()


def metrics(P, Ys, idx, thr=THRESH):
    # pooled pixel metrics over the subset (same definition as the training notebooks' test metrics)
    # plus the mean of per-image IoU
    pred = P[idx].float() >= thr
    gt = Ys[idx]
    tp_i = (pred & gt).flatten(1).sum(1).double()
    fp_i = (pred & ~gt).flatten(1).sum(1).double()
    fn_i = (~pred & gt).flatten(1).sum(1).double()
    tn_i = (~pred & ~gt).flatten(1).sum(1).double()
    tp, fp, fn, tn = tp_i.sum(), fp_i.sum(), fn_i.sum(), tn_i.sum()
    eps = 1e-8
    iou = tp / (tp + fp + fn + eps)
    return {
        'IoU': iou.item(),
        'mIoU': ((iou + tn / (tn + fp + fn + eps)) / 2).item(),
        'Dice': (2 * tp / (2 * tp + fp + fn + eps)).item(),
        'Precision': (tp / (tp + fp + eps)).item(),
        'Recall': (tp / (tp + fn + eps)).item(),
        'img-IoU': (tp_i / (tp_i + fp_i + fn_i).clamp(min=1)).mean().item(),
    }


def per_image_iou(P, Ys, thr=THRESH):
    pred = P.float() >= thr
    return ((pred & Ys).flatten(1).sum(1).double() / (pred | Ys).flatten(1).sum(1).double().clamp(min=1)).numpy()


def side_by_side(P_by_ds, subset='All', cols=('IoU', 'Dice', 'Precision', 'Recall')):
    # rows = models, columns = (dataset, metric)
    parts = {}
    for dname in DNAMES:
        d = DS[dname]
        parts[f"{dname} ({len(d['subsets'][subset])})"] = pd.DataFrame(
            {name: metrics(P_by_ds[dname][name], d['Y'], d['subsets'][subset]) for name in nets}).T[list(cols)]
    return pd.concat(parts, axis=1)


nets = {}
for name, cls, fname, _ in MODEL_SPECS:
    if name in ckpt_paths:
        nets[name] = build_model(cls, ckpt_paths[name])
        print(f'loaded {name:27s} {sum(p.numel() for p in nets[name].parameters()):>12,} params')
MNAMES = list(nets)

## 5. Zero-shot: BSP vs BSP augmented (main result)
**Sanity checks:**
* The baseline on the 13 BSP test images should give about **IoU 0.3559** (P 0.7483, R 0.4043), the Experiment 2 value in `experimental results.docx`.
* The 43 originals inside BSP augmented are the same images as BSP, so their scores must match BSP exactly.

In [ ]:
P_zero = {dname: {name: predict_probs(net, DS[dname]['X']) for name, net in nets.items()} for dname in DNAMES}

zs = side_by_side(P_zero)
zs.insert(0, ('Google test (in-domain)', 'IoU'), [GOOGLE_IOU[m] for m in zs.index])
for dname in DNAMES:
    lbl = f"{dname} ({len(DS[dname]['subsets']['All'])})"
    zs[('Gap: Google minus', dname)] = zs[('Google test (in-domain)', 'IoU')] - zs[(lbl, 'IoU')]
zs.to_csv(f'{OUT_DIR}/zero_shot_side_by_side.csv')
print('Zero-shot, all images')
display(zs)

# per-subset IoU for both datasets
rows = []
for dname in DNAMES:
    d = DS[dname]
    for s, idx in d['subsets'].items():
        for name in MNAMES:
            rows.append({'Dataset': dname, 'Subset': f'{s} ({len(idx)})', 'Model': name,
                         **metrics(P_zero[dname][name], d['Y'], idx)})
df_zero = pd.DataFrame(rows)
df_zero.to_csv(f'{OUT_DIR}/zero_shot_all_subsets.csv', index=False)
print('Zero-shot IoU by subset')
display(df_zero.pivot_table(index='Model', columns=['Dataset', 'Subset'], values='IoU', sort=False).loc[MNAMES])

# sanity checks
if BASELINE in nets:
    b = metrics(P_zero['BSP'][BASELINE], DS['BSP']['Y'], DS['BSP']['subsets']['Test 13'])
    print(f"Sanity 1 (baseline, BSP Test 13): IoU {b['IoU']:.4f}  P {b['Precision']:.4f}  R {b['Recall']:.4f}"
          f"   expected ~0.3559 / 0.7483 / 0.4043")
name0 = MNAMES[0]
a = metrics(P_zero['BSP augmented'][name0], DS['BSP augmented']['Y'], DS['BSP augmented']['subsets']['Originals (43)'])['IoU']
b = metrics(P_zero['BSP'][name0], DS['BSP']['Y'], DS['BSP']['subsets']['All'])['IoU']
print(f'Sanity 2 ({name0}): BSP-aug originals IoU {a:.4f} vs BSP IoU {b:.4f}  ->', 'OK' if abs(a - b) < 1e-3 else 'MISMATCH')

# grouped bar chart
fig, ax = plt.subplots(figsize=(11, 4.5))
w = 0.38
xs = np.arange(len(MNAMES))
for k, dname in enumerate(DNAMES):
    vals = [metrics(P_zero[dname][m], DS[dname]['Y'], DS[dname]['subsets']['All'])['IoU'] for m in MNAMES]
    bars = ax.bar(xs + (k - 0.5) * w, vals, w, label=f"{dname} ({len(DS[dname]['meta'])})")
    ax.bar_label(bars, fmt='%.3f', fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels(MNAMES, rotation=15, ha='right')
ax.set_ylabel('IoU'); ax.set_title('Zero-shot IoU (trained on Google only)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/zero_shot_side_by_side.png', dpi=130, bbox_inches='tight'); plt.show()

## 6. BSP augmented: which transformations hurt?
Every augmented copy comes from one of the 20 BSP **training** originals. A fair comparison is therefore each augmentation type against **those same 20 source images** (first column). A clear drop for rotations or blur means the model is not robust to that change. Rotation copies also have black corners.

In [ ]:
d = DS['BSP augmented']
m = d['meta']
groups = {'source (20 train originals)': idx_of((m.aug == 'original') & (m.split == 'train'))}
for t in AUG_TAGS:
    groups[AUG_LABEL[t]] = idx_of(m.aug == t)
rows = []
for name in MNAMES:
    row = {'Model': name}
    for g, idx in groups.items():
        row[f'{g} ({len(idx)})'] = metrics(P_zero['BSP augmented'][name], d['Y'], idx)['IoU']
    rows.append(row)
df_aug = pd.DataFrame(rows).set_index('Model')
src = df_aug.columns[0]
df_aug_delta = df_aug.drop(columns=src).sub(df_aug[src], axis=0)
df_aug.to_csv(f'{OUT_DIR}/zero_shot_by_augmentation.csv')
print('Zero-shot IoU per augmentation type')
display(df_aug)
print('Change vs the 20 source originals (negative = the transformation hurts)')
display(df_aug_delta)

## 7. Label-free test-time methods, side by side
* **TTA**: average of 8 flips and rotations.
* **AdaBN**: each model's BatchNorm statistics are re-estimated on the evaluated dataset's own images, *without labels*. This is transductive but uses no annotations. Statistics are computed separately for BSP and for BSP augmented.

In [ ]:
P_tta, P_ada, P_ada_tta = {}, {}, {}
for dname in DNAMES:
    Xd = DS[dname]['X']
    P_tta[dname] = {name: predict_probs(net, Xd, tta=True) for name, net in nets.items()}
    P_ada[dname], P_ada_tta[dname] = {}, {}
    for name, net in nets.items():
        m_ad = adabn(net, Xd)
        P_ada[dname][name] = predict_probs(m_ad, Xd)
        P_ada_tta[dname][name] = predict_probs(m_ad, Xd, tta=True)
        del m_ad
        if device.type == 'cuda':
            torch.cuda.empty_cache()

METHODS = {'Zero-shot': P_zero, 'TTA': P_tta, 'AdaBN': P_ada, 'AdaBN + TTA': P_ada_tta}
rows = []
for meth, PD in METHODS.items():
    for dname in DNAMES:
        d = DS[dname]
        for name in MNAMES:
            rows.append({'Method': meth, 'Dataset': dname, 'Model': name, **metrics(PD[dname][name], d['Y'], d['subsets']['All'])})
df_tt = pd.DataFrame(rows)
df_tt.to_csv(f'{OUT_DIR}/test_time_methods_side_by_side.csv', index=False)

for metric in ('IoU', 'Precision', 'Recall'):
    print(f'{metric}, all images')
    display(df_tt.pivot_table(index='Model', columns=['Method', 'Dataset'], values=metric, sort=False).loc[MNAMES])

## 8. Input-scale sweep, side by side (diagnoses a panel-size gap)
The image is resized by the factor before the model, and the prediction is resized back to 256. A clear peak away from 1.0 means Bengaluru panels appear at a different size than the Google panels the models were trained on. The scale is not *chosen* here; this is a diagnostic only.

In [ ]:
SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
rows = []
for dname in DNAMES:
    d = DS[dname]
    for name, net in nets.items():
        for s in SCALES:
            P = P_zero[dname][name] if s == 1.0 else predict_probs(net, d['X'], scale=s)
            rows.append({'Dataset': dname, 'Model': name, 'Scale': s, **metrics(P, d['Y'], d['subsets']['All'])})
            del P
df_scale = pd.DataFrame(rows)
df_scale.to_csv(f'{OUT_DIR}/scale_sweep_side_by_side.csv', index=False)
print('Zero-shot IoU vs input scale')
display(df_scale.pivot_table(index='Model', columns=['Dataset', 'Scale'], values='IoU', sort=False).loc[MNAMES])

fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), sharex=True)
for c, dname in enumerate(DNAMES):
    for r, metric in enumerate(('IoU', 'Recall')):
        a = axes[r, c]
        for name in MNAMES:
            dd = df_scale[(df_scale.Dataset == dname) & (df_scale.Model == name)]
            a.plot(dd.Scale, dd[metric], marker='o', label=name)
        a.set_xscale('log', base=2); a.set_xticks(SCALES); a.set_xticklabels(SCALES)
        a.axvline(1.0, color='grey', ls='--', lw=0.8); a.grid(alpha=0.3)
        a.set_title(f"{dname} ({len(DS[dname]['meta'])}): zero-shot {metric} vs input scale")
        if r == 1:
            a.set_xlabel('input scale factor')
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/scale_sweep_side_by_side.png', dpi=130, bbox_inches='tight'); plt.show()

## 9. Diagnostics: why does zero-shot fail?
### 9a. Panel-level detection by panel size, side by side
Each connected region of the ground-truth mask is one "panel region". It counts as **detected** if at least 50% of its pixels are predicted. Size bins are the tertiles of the BSP panel-region sizes, the same bins for both datasets. **FP blobs** are predicted regions that touch no ground-truth panel. **pred/GT area** below 1 means the model under-segments.

In [ ]:
STRUCT = np.ones((3, 3), dtype=int)


def region_sizes(Ys):
    out = []
    for y in Ys.numpy():
        lab, n = ndimage.label(y, STRUCT)
        if n:
            out += list(ndimage.sum(y, lab, range(1, n + 1)))
    return np.array(out)

gt_sizes_bsp = region_sizes(DS['BSP']['Y'])
q1, q2 = np.quantile(gt_sizes_bsp, [1 / 3, 2 / 3])
BINS = [('small', 0, q1), ('medium', q1, q2), ('large', q2, np.inf)]
print(f'BSP: {len(gt_sizes_bsp)} panel regions in {len(DS["BSP"]["Y"])} images; size tertiles {q1:.0f} / {q2:.0f} px (256x256 frame)')


def component_report(P, Ys, thr=THRESH, min_cover=0.5):
    det = {b[0]: [0, 0] for b in BINS}
    fp_blobs = 0
    predb = (P.float() >= thr)
    for p, y in zip(predb.numpy(), Ys.numpy()):
        lab, n = ndimage.label(y, STRUCT)
        if n:
            idx = range(1, n + 1)
            size = ndimage.sum(y, lab, idx)
            cover = ndimage.sum(p, lab, idx) / size
            for s, c in zip(size, cover):
                for bname, lo, hi in BINS:
                    if lo <= s < hi:
                        det[bname][0] += int(c >= min_cover)
                        det[bname][1] += 1
                        break
        plab, pn = ndimage.label(p, STRUCT)
        if pn:
            fp_blobs += int((ndimage.sum(y, plab, range(1, pn + 1)) == 0).sum())
    out = {f'recall {b} ({det[b][1]})': det[b][0] / max(det[b][1], 1) for b in det}
    out['FP blobs / image'] = fp_blobs / len(P)
    out['pred / GT area'] = predb.float().mean().item() / Ys.float().mean().item()
    return out

parts = {}
for dname in DNAMES:
    parts[dname] = pd.DataFrame({name: component_report(P_zero[dname][name], DS[dname]['Y']) for name in MNAMES}).T
df_comp = pd.concat(parts, axis=1)
df_comp.to_csv(f'{OUT_DIR}/panel_level_zero_shot_side_by_side.csv')
display(df_comp)

### 9b. Google vs Bengaluru: panel size and image appearance
If Bengaluru panel regions are much smaller or larger (in pixels at 256x256) than Google ones, that is a **scale gap**. The suggested rescale factor is sqrt(median Google area / median BSP area); compare it with the peak in section 8. The colour statistics show the **appearance gap**. Only the 43 BSP originals are used here; the augmented copies have the same panels.

In [ ]:
def appearance_stats(rgb):
    hsv = np.array(Image.fromarray(rgb).convert('HSV')).astype(np.float32)
    g = rgb.astype(np.float32).mean(axis=2)
    return {'brightness': g.mean(), 'contrast': g.std(), 'saturation': hsv[..., 1].mean(),
            'R': rgb[..., 0].mean(), 'G': rgb[..., 1].mean(), 'B': rgb[..., 2].mean()}

app_bsp = pd.DataFrame([appearance_stats(r) for r in DS['BSP']['RGB']])
rows = [{'Dataset': f"BSP ({len(DS['BSP']['Y'])})", 'panel regions / image': len(gt_sizes_bsp) / len(DS['BSP']['Y']),
         'median region area (px)': np.median(gt_sizes_bsp), 'mean panel fraction': DS['BSP']['meta'].panel_frac.mean(),
         **app_bsp.mean().to_dict()}]

gt_sizes_g = app_g = None
if GOOGLE_ROOT is not None:
    g_img_dir, g_mask_dir = split_dirs(GOOGLE_ROOT, 'img', 'mask')['test']
    g_masks = {os.path.splitext(f)[0].replace('_gt', ''): f for f in os.listdir(g_mask_dir) if f.lower().endswith(IMG_EXTS)}
    sizes, app, frac = [], [], []
    for f in sorted(os.listdir(g_img_dir)):
        st = os.path.splitext(f)[0]
        if st not in g_masks:
            continue
        rgb = np.array(Image.open(os.path.join(g_img_dir, f)).convert('RGB').resize(IMG_SIZE))
        y = np.array(Image.open(os.path.join(g_mask_dir, g_masks[st])).convert('L').resize(IMG_SIZE, Image.NEAREST)) > 0
        lab, n = ndimage.label(y, STRUCT)
        if n:
            sizes += list(ndimage.sum(y, lab, range(1, n + 1)))
        app.append(appearance_stats(rgb)); frac.append(y.mean())
    gt_sizes_g, app_g = np.array(sizes), pd.DataFrame(app)
    rows.append({'Dataset': f'Google test ({len(frac)})', 'panel regions / image': len(gt_sizes_g) / len(frac),
                 'median region area (px)': np.median(gt_sizes_g), 'mean panel fraction': np.mean(frac),
                 **app_g.mean().to_dict()})

dom = pd.DataFrame(rows).set_index('Dataset')
dom.to_csv(f'{OUT_DIR}/domain_statistics.csv')
display(dom)

if gt_sizes_g is not None:
    print(f'Suggested BSP input rescale from median panel area: x{math.sqrt(np.median(gt_sizes_g) / np.median(gt_sizes_bsp)):.2f}'
          '  (compare with the peak of the scale sweep in section 8)')
    fig, ax = plt.subplots(1, 4, figsize=(20, 4))
    bins = np.logspace(0, np.log10(max(gt_sizes_g.max(), gt_sizes_bsp.max()) + 1), 40)
    ax[0].hist(gt_sizes_g, bins=bins, alpha=0.6, density=True, label='Google test')
    ax[0].hist(gt_sizes_bsp, bins=bins, alpha=0.6, density=True, label='BSP')
    ax[0].set_xscale('log'); ax[0].set_title('panel-region area (px at 256x256)'); ax[0].legend()
    for a, col in zip(ax[1:], ('brightness', 'contrast', 'saturation')):
        a.hist(app_g[col], bins=30, alpha=0.6, density=True, label='Google test')
        a.hist(app_bsp[col], bins=30, alpha=0.6, density=True, label='BSP')
        a.set_title(col); a.legend()
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/domain_statistics.png', dpi=130, bbox_inches='tight'); plt.show()

### 9c. Per-image IoU (zero-shot)

In [ ]:
per_img = {}
for dname in DNAMES:
    t = DS[dname]['meta'][['name', 'split', 'layout', 'aug', 'source', 'panel_frac']].copy()
    for name in MNAMES:
        t[name] = per_image_iou(P_zero[dname][name], DS[dname]['Y'])
    per_img[dname] = t
    t.to_csv(f"{OUT_DIR}/per_image_iou_{dname.replace(' ', '_')}.csv", index=False)
print('BSP (43), sorted by the first model (worst first)')
display(per_img['BSP'].drop(columns=['aug', 'source']).sort_values(MNAMES[0]))

## 10. Overlays
Colours: **green = correct panel pixels (TP)**, **red = false positive**, **blue = missed panel (FN)**. The number above each panel is that image's IoU.
* BSP: all 43 images, one figure per layout (zero-shot).
* BSP augmented: the 8 versions of two training images, which shows how the predictions change under each transformation.

In [ ]:
def overlay(rgb, pred, gt):
    out = rgb.astype(np.float32) / 255.0 * 0.55
    for msk, c in ((pred & gt, (0.1, 0.95, 0.1)), (pred & ~gt, (1.0, 0.15, 0.15)), (~pred & gt, (0.2, 0.45, 1.0))):
        out[msk] = 0.35 * out[msk] + 0.65 * np.array(c)
    return np.clip(out, 0, 1)


def show_grid(dname, P_dict, idx, title, fname, thr=THRESH):
    d = DS[dname]
    idx = [int(i) for i in idx]
    cols = 2 + len(P_dict)
    fig, axes = plt.subplots(len(idx), cols, figsize=(2.4 * cols, 2.5 * len(idx)), squeeze=False)
    for r, i in enumerate(idx):
        gt = d['Y'][i].numpy()
        axes[r, 0].imshow(d['RGB'][i]); axes[r, 0].set_title(d['meta']['name'].iloc[i], fontsize=7)
        axes[r, 1].imshow(gt, cmap='gray'); axes[r, 1].set_title('ground truth', fontsize=8)
        for c, (name, P) in enumerate(P_dict.items()):
            pred = P[i].float().numpy() >= thr
            iou = (pred & gt).sum() / max((pred | gt).sum(), 1)
            axes[r, c + 2].imshow(overlay(d['RGB'][i], pred, gt))
            axes[r, c + 2].set_title(f'{name}\nIoU {iou:.2f}', fontsize=7)
        for a in axes[r]:
            a.axis('off')
    fig.suptitle(title, fontsize=12, y=1.0)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{fname}', dpi=100, bbox_inches='tight'); plt.show()


for lay in ('Haralur', 'HSR', 'Krystal'):
    show_grid('BSP', P_zero['BSP'], DS['BSP']['subsets'][lay], f'BSP {lay}: zero-shot', f'overlay_BSP_{lay}.png')

ma = DS['BSP augmented']['meta']
for src_name in sorted(ma[(ma.aug != 'original')].source.unique())[:2]:
    idx = idx_of(ma.source == src_name)
    show_grid('BSP augmented', P_zero['BSP augmented'], idx, f'BSP augmented: all versions of {src_name} (zero-shot)',
              f'overlay_BSPaug_{src_name}.png')

## 11. Headline table and saved outputs

In [ ]:
rows = []
for name in MNAMES:
    row = {'Model': name, 'Google (in-domain)': GOOGLE_IOU[name]}
    for meth, PD in METHODS.items():
        for dname in DNAMES:
            d = DS[dname]
            row[(meth, dname)] = metrics(PD[dname][name], d['Y'], d['subsets']['All'])['IoU']
    rows.append(row)
headline = pd.DataFrame(rows).set_index('Model')
headline.columns = [c if isinstance(c, str) else f'{c[0]} | {c[1]}' for c in headline.columns]
headline.to_csv(f'{OUT_DIR}/headline_iou.csv')
print('IoU on all images (BSP = 43, BSP augmented = 183). No labels used anywhere.')
display(headline)

if BASELINE in nets:
    print('\nZero-shot IoU change vs baseline ResNet-UNet  (BSP / BSP augmented):')
    for name in MNAMES:
        if name != BASELINE:
            print(f"  {name:27s} {headline.loc[name, 'Zero-shot | BSP'] - headline.loc[BASELINE, 'Zero-shot | BSP']:+.4f}"
                  f"  /  {headline.loc[name, 'Zero-shot | BSP augmented'] - headline.loc[BASELINE, 'Zero-shot | BSP augmented']:+.4f}")
print('\nNote: BSP has only 43 images from 3 layouts, and BSP augmented adds no new scenes, so differences below'
      ' ~0.01-0.02 IoU are within noise. Check the per-layout columns in section 5 before calling a model better.')

print()
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{OUT_DIR}/{f}  ({os.path.getsize(os.path.join(OUT_DIR, f)) / 1024:.0f} KB)')